<a href="https://colab.research.google.com/github/profliuhao/CSIT599/blob/main/CSIT599_Online_lab2_image_classification_cnns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Image Classification with CNNs (PyTorch)

**Module 2: Advanced Neural Networks and Computer Vision**

In this lab you will build a **Convolutional Neural Network (CNN)** in PyTorch, train it
on the **CIFAR-10** image dataset, and then compare your hand-built model against a
**pre-trained ResNet-18** adapted with transfer learning.

```
image (3 x 32 x 32) --> [Conv blocks] --> [Flatten] --> [Linear] --> class scores (10)
```

## What you will do
You only need to fill in **five** short pieces of code, each marked with a `TODO`:
  1. the **convolutional blocks** of `SmallCNN`
  2. the **forward pass** of `SmallCNN`
  3. one **training step** (forward → loss → backward → update)
  4. the **evaluation** function (accuracy on a data loader)
  5. **transfer learning**: adapt a pre-trained ResNet-18 to CIFAR-10

Everything else — data loading, the training driver, plots, and the comparison table —
is already written for you.

## How to use this file
* Recommended: open in **Google Colab** and switch the runtime to **GPU**
  (`Runtime > Change runtime type > T4 GPU`). CPU works too, just slower (~10 min).
* Fill in each `TODO`, then run the cells from top to bottom (or "Run All").
* The lab trains on a **subset** of CIFAR-10 so it finishes in a few minutes.

## Setup — imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms

SEED = 42
BATCH_SIZE = 128
EPOCHS_CNN = 6          # epochs for your SmallCNN
EPOCHS_TRANSFER = 2     # epochs for the ResNet-18 head
TRAIN_SUBSET = 10_000   # use 10k of the 50k training images so the lab runs fast
TEST_SUBSET = 2_000
LEARNING_RATE = 1e-3

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Section 1 — The data *(provided)*

CIFAR-10: 60,000 color images (32x32), 10 classes. We normalize with the dataset's
per-channel mean/std and take a fixed random subset so training is quick.

In [ ]:
norm = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    norm,
])
test_tf = transforms.Compose([transforms.ToTensor(), norm])

train_full = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
test_full = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

rng = np.random.RandomState(SEED)
train_idx = rng.choice(len(train_full), TRAIN_SUBSET, replace=False)
test_idx = rng.choice(len(test_full), TEST_SUBSET, replace=False)
train_ds, test_ds = Subset(train_full, train_idx), Subset(test_full, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)

CLASSES = train_full.classes
print("classes:", CLASSES)
print("train:", len(train_ds), " test:", len(test_ds))

In [ ]:
# Peek at a few training images (provided).
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab in zip(axes, imgs[:8], labels[:8]):
    img = img.permute(1, 2, 0).numpy() * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
    ax.imshow(np.clip(img, 0, 1)); ax.set_title(CLASSES[lab], fontsize=8); ax.axis("off")
plt.show()

## Section 2 — Your CNN

**TODO 1:** define the three convolutional blocks. Each block is:

```
Conv2d(in_ch, out_ch, kernel_size=3, padding=1) -> BatchNorm2d(out_ch) -> ReLU -> MaxPool2d(2)
```

| block | in channels | out channels | spatial size after pool |
|-------|------------|--------------|--------------------------|
| 1     | 3          | 32           | 16 x 16                  |
| 2     | 32         | 64           | 8 x 8                    |
| 3     | 64         | 128          | 4 x 4                    |

**TODO 2:** implement `forward`: pass `x` through the three blocks, flatten to
shape `(batch, 128*4*4)`, apply dropout, then the final linear layer.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes: int = 10):
        super().__init__()
        # ===== TODO 1: define self.block1, self.block2, self.block3 =====
        # Hint: use nn.Sequential(nn.Conv2d(...), nn.BatchNorm2d(...), nn.ReLU(), nn.MaxPool2d(2))
        self.block1 = None  # replace
        self.block2 = None  # replace
        self.block3 = None  # replace
        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(128 * 4 * 4, n_classes)

    def forward(self, x):
        # ===== TODO 2: blocks -> flatten -> dropout -> fc =====
        # Hint: after the blocks use torch.flatten(x, 1) to keep the batch dimension.
        raise NotImplementedError("TODO 2: implement the forward pass, then delete this line.")

model = SmallCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"parameters: {n_params:,}")

# Quick shape check: a fake batch must produce (8, 10) scores.
with torch.no_grad():
    out = model(torch.zeros(8, 3, 32, 32, device=device))
assert out.shape == (8, 10), f"expected (8, 10), got {tuple(out.shape)}"
print("shape check passed ✔")

## Section 3 — Training step and evaluation

**TODO 3:** one optimization step on a single batch:
1. move `imgs`/`labels` to `device`,
2. `optimizer.zero_grad()`,
3. forward pass → `logits`,
4. `loss = criterion(logits, labels)`,
5. `loss.backward()` then `optimizer.step()`,
6. return the loss as a Python float (`loss.item()`).

**TODO 4:** accuracy over a loader: in `torch.no_grad()` mode and with `net.eval()`,
count how many `argmax` predictions match the labels and divide by the total.

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_step(net, imgs, labels, optimizer):
    """Run ONE gradient update and return the batch loss (float)."""
    net.train()
    # ===== TODO 3: implement the training step =====
    raise NotImplementedError("TODO 3: implement the training step, then delete this line.")

def evaluate(net, loader):
    """Return accuracy of `net` on all batches of `loader` (a float in [0, 1])."""
    net.eval()
    correct, total = 0, 0
    # ===== TODO 4: loop over loader, count correct argmax predictions =====
    raise NotImplementedError("TODO 4: implement evaluation, then delete this line.")

## Section 4 — Train your CNN *(provided)*

In [ ]:
def fit(net, epochs, lr=LEARNING_RATE, tag="model"):
    optimizer = torch.optim.Adam((p for p in net.parameters() if p.requires_grad), lr=lr)
    history = {"loss": [], "test_acc": []}
    for epoch in range(1, epochs + 1):
        losses = [train_step(net, imgs, labels, optimizer) for imgs, labels in train_loader]
        acc = evaluate(net, test_loader)
        history["loss"].append(np.mean(losses))
        history["test_acc"].append(acc)
        print(f"[{tag}] epoch {epoch:2d}/{epochs}  loss={np.mean(losses):.4f}  test_acc={acc:.3f}")
    return history

hist_cnn = fit(model, EPOCHS_CNN, tag="SmallCNN")

In [ ]:
# Learning curves (provided).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist_cnn["loss"]); ax[0].set_title("SmallCNN training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(hist_cnn["test_acc"]); ax[1].set_title("SmallCNN test accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## Section 5 — Transfer learning with ResNet-18

Training a deep network from scratch on 10k images only gets you so far. Instead we
take a **ResNet-18 pre-trained on ImageNet** and reuse its convolutional features.

**TODO 5:**
1. load `torchvision.models.resnet18(weights="IMAGENET1K_V1")`,
2. **freeze** every parameter (`p.requires_grad = False`),
3. replace the final fully connected layer `resnet.fc` with a fresh
   `nn.Linear(resnet.fc.in_features, 10)` (a new layer is trainable by default).

The data pipeline below resizes CIFAR images to 224x224, the input size ResNet expects.

In [ ]:
resize_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),  # ImageNet stats
])
train_r = Subset(torchvision.datasets.CIFAR10("./data", train=True, transform=resize_tf), train_idx[:4000])
test_r = Subset(torchvision.datasets.CIFAR10("./data", train=False, transform=resize_tf), test_idx)
train_loader_r = DataLoader(train_r, batch_size=64, shuffle=True, num_workers=2)
test_loader_r = DataLoader(test_r, batch_size=128, shuffle=False, num_workers=2)

# ===== TODO 5: build `resnet` (pre-trained, frozen, new 10-class head) =====
resnet = None  # replace with the three steps above
raise NotImplementedError("TODO 5: build the transfer-learning model, then delete this line.")
resnet = resnet.to(device)

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"trainable parameters: {trainable:,} (only the new head)")

In [ ]:
# Train just the head — note we reuse YOUR train_step/evaluate via fit(). (provided)
train_loader, test_loader = train_loader_r, test_loader_r   # point fit() at the resized data
hist_resnet = fit(resnet, EPOCHS_TRANSFER, lr=3e-4, tag="ResNet-18")

## Section 6 — Results — compare the two models *(provided)*

In [ ]:
acc_cnn, acc_resnet = hist_cnn["test_acc"][-1], hist_resnet["test_acc"][-1]
print(f"{'model':<22}{'trained params':>16}{'test accuracy':>15}")
print(f"{'SmallCNN (scratch)':<22}{n_params:>16,}{acc_cnn:>15.3f}")
print(f"{'ResNet-18 (transfer)':<22}{trainable:>16,}{acc_resnet:>15.3f}")

# Per-class accuracy for the better model.
best = resnet if acc_resnet >= acc_cnn else model
loader = test_loader_r if best is resnet else test_loader
correct = np.zeros(10); total = np.zeros(10)
best.eval()
with torch.no_grad():
    for imgs, labels in loader:
        preds = best(imgs.to(device)).argmax(1).cpu()
        for c in range(10):
            m = labels == c
            correct[c] += (preds[m] == c).sum().item(); total[c] += m.sum().item()
plt.figure(figsize=(8, 3))
plt.bar(CLASSES, correct / np.maximum(total, 1))
plt.title("Per-class accuracy (best model)"); plt.xticks(rotation=45); plt.ylim(0, 1)
plt.tight_layout(); plt.show()

## Checklist before you submit

* [ ] All cells run top-to-bottom without errors, outputs visible.
* [ ] The shape check after `SmallCNN` passes.
* [ ] SmallCNN reaches at least **~65%** test accuracy after 6 epochs.
* [ ] ResNet-18 transfer reaches at least **~80%** test accuracy after 2 epochs.
* [ ] The comparison table and per-class accuracy plot are shown.

**Submit your completed notebook (.ipynb) with all outputs visible.**

*Where you'll see this again:* HW 1 asks you to train and compare CNN architectures on
a benchmark dataset — the training loop, evaluation, and transfer-learning patterns you
wrote here are exactly the tools you'll need.